# Historical Data Import, Reconfiguration, and Clean

#### This notebook covers the extraction of historical rosters and driver/contructor points and prices from 2023 through 2026
#### Data that I have inputed (rawHistPointandPrice.xlsx) was manually pulled from `https://f1fantasytools.com/statistics`


FUTURE NOTE:

Need to be able to ingest 2026 data eventually



### Libraries

In [163]:
import os
from pathlib import Path
import requests
import pandas as pd
import importlib



#Loading the Placement Table Module
import sys
sys.path.append("/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/notebooks/cleaning")

import placementTable as pt
importlib.reload(pt)


Resolved file path: /Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/data/clean/race_session_meeting_info.csv
Exists: True


<module 'placementTable' from '/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/notebooks/cleaning/placementTable.py'>

### Setting working directory to make it easy for file path management

In [164]:
working_directory = Path.cwd().parent.parent
print(working_directory)

file_path = working_directory / "data" / "raw" / "rawHistPointAndPrice.xlsx"

/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


### Importing Sheets

In [165]:
sheet_names =["2023DriverPrice", "2023ConstructorPrice", "2023DriverPoints","2023ConstructorPoints","2023DriverRoster", 
              "2024DriverPrice", "2024ConstructorPrice", "2024DriverPoints", "2024ConstructorPoints", "2024DriverRoster",
              "2025DriverPrice", "2025ConstructorPrice","2025DriverPoints", "2025ConstructorPoints",  "2025DriverRoster",
               "2026DriverRoster", "2026DriverPrice", "2026DriverPoints", "2026ConstructorPrice", "2026ConstructorPoints"]



dfs = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names}

# access one
print(dfs["2023DriverPoints"].head())




  Unnamed: 0     1     2     3     4     5     7     8     9    10  ...    14  \
0        VER  35.0  61.0  36.0  37.0  64.0  35.0  46.0  35.0  56.0  ...  46.0   
1        PER  28.0  36.0  52.0  61.0  28.0  10.0  27.0  28.0  51.0  ...  30.0   
2        HAM  19.0  16.0  29.0  19.0  24.0  28.0  39.0  23.0  31.0  ...  29.0   
3        NOR  -1.0   7.0  19.0   0.0   1.0   6.0  -2.0   2.0  29.0  ...  25.0   
4        ALO  39.0  23.0  23.0  27.0  25.0  27.0  14.0  27.0  28.0  ...  62.0   

     15    16    17    18    19    20    21    22    23  
0  38.0  25.0  47.0  62.0  53.0  42.0  45.0  44.0  47.0  
1  30.0  16.0 -11.0  -3.0  32.0 -10.0  37.0  35.0  30.0  
2  17.0  39.0  20.0   6.0  -4.0  41.0  13.0  26.0  14.0  
3   9.0  30.0  31.0  41.0  46.0  48.0  57.0 -14.0  22.0  
4   5.0  -1.0  16.0  25.0  -9.0 -16.0  39.0  12.0  16.0  

[5 rows x 23 columns]


For each sheet we will need to unpivot them so that they are long and not wide format
Each sheet will need to have col headers of: YEAR, RACE, DRIVER, POINTS/PRICE

#### Sheet Organization

In [166]:
driver_price_sheets = ["2023DriverPrice", "2024DriverPrice", "2025DriverPrice", "2026DriverPrice"]
driver_points_sheets = ["2023DriverPoints", "2024DriverPoints", "2025DriverPoints", "2026DriverPoints"]

con_price_sheets = ["2023ConstructorPrice", "2024ConstructorPrice", "2025ConstructorPrice", "2026ConstructorPrice"]
con_points_sheets = ["2023ConstructorPoints", "2024ConstructorPoints", "2025ConstructorPoints", "2026ConstructorPoints"]


driver_roster_sheets = ["2023DriverRoster", "2024DriverRoster", "2025DriverRoster", "2026DriverRoster"]

### Two functions to help unpivot/melt the existing pivots in each sheet.

In [167]:

def standardize_headers(df: pd.DataFrame, first_col_name: str) -> pd.DataFrame:
    df = df.copy()

    # Dropping unnamed cols
    unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
    for c in unnamed_cols:
        if df[c].isna().all():
            df = df.drop(columns=c)

    # renaming first col to first_col_name
    df = df.rename(columns={df.columns[0]: first_col_name})

    # Clean race headers
    cleaned_cols = [first_col_name]
    for c in df.columns[1:]:
        try:
            cleaned_cols.append(int(float(str(c).strip())))
        except ValueError:
            cleaned_cols.append(str(c).strip())

    df.columns = cleaned_cols
    return df


def convert_to_long(sheets: dict, var_name: str, value_name: str, type: str):
    longs = []

    for sheet_name, df in sheets.items():
        df_clean = standardize_headers(df, first_col_name=type)

        temp = (
            df_clean
            .melt(id_vars=type, var_name=var_name, value_name=value_name)
            .assign(source_sheet=sheet_name)
        )
        longs.append(temp)

    return pd.concat(longs, ignore_index=True)



### Melting each sheet to long format

In [168]:
driver_price_dfs  = {k: dfs[k] for k in driver_price_sheets}
driver_points_dfs = {k: dfs[k] for k in driver_points_sheets}

con_price_dfs = {k: dfs[k] for k in con_price_sheets}
con_points_dfs = {k: dfs[k] for k in con_points_sheets}

driver_roster_dfs = {k: dfs[k] for k in driver_roster_sheets}

driver_price_long  = convert_to_long(driver_price_dfs,  var_name="race", value_name="price",  type = "driver")
driver_points_long  = convert_to_long(driver_points_dfs,  var_name="race", value_name="points",  type = "driver")

con_price_long = convert_to_long(con_price_dfs, var_name="race", value_name="price", type= "constructor")
con_points_long = convert_to_long(con_points_dfs, var_name="race", value_name="points", type= "constructor")

driver_roster_long = convert_to_long(driver_roster_dfs, var_name="race", value_name="constructor", type='driver')

In [169]:
driver_price_long.head()

,driver,race,price,source_sheet
0,VER,1,26.9,2023DriverPrice
1,HAM,1,23.7,2023DriverPrice
2,LEC,1,21.2,2023DriverPrice
3,PER,1,18.0,2023DriverPrice
4,RUS,1,18.6,2023DriverPrice


In [170]:
driver_points_long.head()

,driver,race,points,source_sheet
0,VER,1,35.0,2023DriverPoints
1,PER,1,28.0,2023DriverPoints
2,HAM,1,19.0,2023DriverPoints
3,NOR,1,-1.0,2023DriverPoints
4,ALO,1,39.0,2023DriverPoints


In [171]:
con_price_long.head()

,constructor,race,price,source_sheet
0,RED,1,27.2,2023ConstructorPrice
1,MER,1,25.1,2023ConstructorPrice
2,FER,1,22.1,2023ConstructorPrice
3,MCL,1,9.1,2023ConstructorPrice
4,ALP,1,10.1,2023ConstructorPrice


In [172]:
con_points_long.head()

,constructor,race,points,source_sheet
0,RED,1,78.0,2023ConstructorPoints
1,MER,1,45.0,2023ConstructorPoints
2,FER,1,31.0,2023ConstructorPoints
3,MCL,1,-16.0,2023ConstructorPoints
4,AST,1,56.0,2023ConstructorPoints


In [173]:
driver_roster_long.head()

,driver,race,constructor,source_sheet
0,VER,1,RBR,2023DriverRoster
1,PER,1,RBR,2023DriverRoster
2,NOR,1,MCL,2023DriverRoster
3,PIA,1,MCL,2023DriverRoster
4,HAM,1,MER,2023DriverRoster


### Slight Feature Engineering and Conversions

In [174]:
dfs = [driver_price_long, driver_points_long, con_price_long, con_points_long, driver_roster_long]

In [175]:
#Extracting out year from source_sheet column string
for df in dfs:
    df["year"] = df["source_sheet"].str[:4].astype(int)
    df = df.drop(columns="source_sheet", inplace=True)

In [176]:
for df in dfs:
    df["race"] = df["race"].astype("int")


    print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1724 entries, 0 to 1723
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   driver  1724 non-null   object 
 1   race    1724 non-null   int64  
 2   price   1487 non-null   float64
 3   year    1724 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 54.0+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1676 entries, 0 to 1675
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   driver  1676 non-null   object 
 1   race    1676 non-null   int64  
 2   points  1465 non-null   float64
 3   year    1676 non-null   int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 52.5+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   constructor  74

### Creating a Race Dimension Table using the Open F1 API

- We have created two functions that allow us to create df's of output from the API at both the meeting and the race session level
- The last function then merges both tables together to create the dimension table. this will be useful when joining to other tables for race session info


- I absolutely am loving how easy it is to work with this API


This API will only work on historical races, and wont be able to input of current week races

In [177]:
def build_meetings_openf1(year: int):
    url = f"https://api.openf1.org/v1/meetings?year={year}"
    meetings = requests.get(url, timeout=30).json()
    df = pd.DataFrame(meetings)

    # Sort by start date and create race number 1..N
    df = df.sort_values("date_start").reset_index(drop=True)
    df["race"] = df.index + 1

    # Extracting Columns
    meetings = df.rename(columns={
        "meeting_name": "race_name",
        "date_start": "weekend_start_date",
        "country_name": "country",
    })[["year", "race", "meeting_key", "race_name", "weekend_start_date", "location", "country_code"]]

    return meetings

def build_sessions_openf1(year:int):
    """
    This will pull each race session

    Ideally we will pull date_start, date_end, and country_code
    """

    url = f"https://api.openf1.org/v1/sessions?year={year}&session_name=Race"
    sessions = requests.get(url, timeout=30).json()
    df = pd.DataFrame(sessions)

    if df.empty:
        return pd.DataFrame(columns=["meeting_key", "race_start_date", "race_end_date"])

    # Keep only what is needed
    sessions = df.rename(columns={
        "date_start": "race_start_date",
        "date_end": "race_end_date",
    })[["meeting_key", "race_start_date", "race_end_date"]].drop_duplicates()

    return sessions


def build_race_dim_open_f1(year:int):
    meetings = build_meetings_openf1(year)
    race_sessions = build_sessions_openf1(year)

    #join
    out = meetings.merge(race_sessions, on="meeting_key", how="left")
    out = out.rename(columns={"race_start_date": "start_date"})

    return out[["year", "race", "race_name", "start_date", "location", "country_code", "meeting_key"]]

In [178]:
race_dim = pd.concat([build_race_dim_open_f1(y) for y in [2023, 2024, 2025, 2026]], ignore_index=True)

In [179]:
#race_dim.head()
race_dim.tail(100)
#race_dim.info()


,year,race,race_name,start_date,location,country_code,meeting_key
0,2023,1,Pre-Season Testing,NaN,Sakhir,BRN,1140
1,2023,2,Bahrain Grand Prix,2023-03-05T15:00:00+00:00,Sakhir,BRN,1141
2,2023,3,Saudi Arabian Grand Prix,2023-03-19T17:00:00+00:00,Jeddah,KSA,1142
3,2023,4,Australian Grand Prix,2023-04-02T05:00:00+00:00,Melbourne,AUS,1143
4,2023,5,Azerbaijan Grand Prix,2023-04-30T11:00:00+00:00,Baku,AZE,1207
...,...,...,...,...,...,...,...
95,2026,22,Mexico City Grand Prix,2026-11-01T20:00:00+00:00,Mexico City,MEX,1298
96,2026,23,São Paulo Grand Prix,2026-11-08T17:00:00+00:00,São Paulo,BRA,1299
97,2026,24,Las Vegas Grand Prix,2026-11-22T04:00:00+00:00,Las Vegas,USA,1300
98,2026,25,Qatar Grand Prix,2026-11-29T16:00:00+00:00,Lusail,QAT,1301


### Need to fix race_num
- dont want to include pre-season testing
- sort by start date and have the min date be 1
- increment race_num until near year is reached

In [180]:
race_dim = race_dim[~race_dim["race_name"].str.contains("test", case=False, na=False)].copy()
race_dim["start_date"] = pd.to_datetime(race_dim["start_date"], utc=True, errors="coerce")
race_dim = race_dim.sort_values(["year", "start_date", "race_name"]).reset_index(drop=True)
race_dim["race"] = race_dim.groupby("year").cumcount() + 1 #renumbering


In [181]:
race_dim.tail(50)

,year,race,race_name,start_date,location,country_code,meeting_key
45,2024,23,Qatar Grand Prix,2024-12-01 16:00:00+00:00,Lusail,QAT,1251
46,2024,24,Abu Dhabi Grand Prix,2024-12-08 13:00:00+00:00,Yas Island,UAE,1252
47,2025,1,Australian Grand Prix,2025-03-16 04:00:00+00:00,Melbourne,AUS,1254
48,2025,2,Chinese Grand Prix,2025-03-23 07:00:00+00:00,Shanghai,CHN,1255
49,2025,3,Japanese Grand Prix,2025-04-06 05:00:00+00:00,Suzuka,JPN,1256
50,2025,4,Bahrain Grand Prix,2025-04-13 15:00:00+00:00,Sakhir,BRN,1257
51,2025,5,Saudi Arabian Grand Prix,2025-04-20 17:00:00+00:00,Jeddah,KSA,1258
52,2025,6,Miami Grand Prix,2025-05-04 20:00:00+00:00,Miami Gardens,USA,1259
53,2025,7,Emilia Romagna Grand Prix,2025-05-18 13:00:00+00:00,Imola,ITA,1260
54,2025,8,Monaco Grand Prix,2025-05-25 13:00:00+00:00,Monaco,MON,1261


### Taking care of Missingness


In [182]:
race_dim[race_dim["start_date"].isna()]

,year,race,race_name,start_date,location,country_code,meeting_key


In [183]:
#dropping all of these as we are not conserned about modeling pre-season testing as we cant see the data from those sessions anyway
# even if we could see that data it is well known that constructors sand bag their performance

In [184]:
race_dim = race_dim.dropna(subset=["start_date"])
race_dim["start_date"].isna().sum()

np.int64(0)

In [185]:
race_dim.tail()

,year,race,race_name,start_date,location,country_code,meeting_key
90,2026,20,Mexico City Grand Prix,2026-11-01 20:00:00+00:00,Mexico City,MEX,1298
91,2026,21,São Paulo Grand Prix,2026-11-08 17:00:00+00:00,São Paulo,BRA,1299
92,2026,22,Las Vegas Grand Prix,2026-11-22 04:00:00+00:00,Las Vegas,USA,1300
93,2026,23,Qatar Grand Prix,2026-11-29 16:00:00+00:00,Lusail,QAT,1301
94,2026,24,Abu Dhabi Grand Prix,2026-12-06 13:00:00+00:00,Yas Marina,UAE,1302


In [186]:
race_dim["start_date"] = pd.to_datetime(race_dim["start_date"], utc=True)
race_dim["month"] = race_dim["start_date"].dt.month
race_dim["start_epoch"] = race_dim["start_date"].astype("int64") // 10**9

In [187]:
race_dim.head()

,year,race,race_name,start_date,location,country_code,meeting_key,month,start_epoch
0,2023,1,Bahrain Grand Prix,2023-03-05 15:00:00+00:00,Sakhir,BRN,1141,3,1678028400
1,2023,2,Saudi Arabian Grand Prix,2023-03-19 17:00:00+00:00,Jeddah,KSA,1142,3,1679245200
2,2023,3,Australian Grand Prix,2023-04-02 05:00:00+00:00,Melbourne,AUS,1143,4,1680411600
3,2023,4,Azerbaijan Grand Prix,2023-04-30 11:00:00+00:00,Baku,AZE,1207,4,1682852400
4,2023,5,Miami Grand Prix,2023-05-07 19:30:00+00:00,Miami,USA,1208,5,1683487800


In [188]:
race_dim.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   year          95 non-null     int64              
 1   race          95 non-null     int64              
 2   race_name     95 non-null     object             
 3   start_date    95 non-null     datetime64[ns, UTC]
 4   location      95 non-null     object             
 5   country_code  95 non-null     object             
 6   meeting_key   95 non-null     int64              
 7   month         95 non-null     int32              
 8   start_epoch   95 non-null     int64              
dtypes: datetime64[ns, UTC](1), int32(1), int64(4), object(3)
memory usage: 6.4+ KB


### Exporting the dimension table on its own if needed later for anything

In [189]:
file_path = working_directory / "data" / "clean" / "race_session_meeting_info.csv"
race_dim.to_csv(file_path)

### Creating the Placement Table
- will use the race_session_meeting_info table for the meeting keys

In [190]:
placement_table = pt.get_placement_df()

ADDED session_key 7953


ADDED session_key 7779
ADDED session_key 7787
ADDED session_key 9070
ADDED session_key 9078
SKIPPING session_key 9086: 404 not found for session_key:9086
ADDED session_key 9094
ADDED session_key 9102
ADDED session_key 9110
ADDED session_key 9118
ADDED session_key 9126
Rate Limited on session_key=9133. Sleeping 2 seconds
Rate Limited on session_key=9133. Sleeping 4 seconds
Rate Limited on session_key=9133. Sleeping 6 seconds
ADDED session_key 9133
ADDED session_key 9141
ADDED session_key 9149
ADDED session_key 9157
ADDED session_key 9165
ADDED session_key 9173
ADDED session_key 9221
ADDED session_key 9213
ADDED session_key 9181
ADDED session_key 9205
ADDED session_key 9189
ADDED session_key 9197
ADDED session_key 9472
ADDED session_key 9480
Rate Limited on session_key=9488. Sleeping 2 seconds
Rate Limited on session_key=9488. Sleeping 4 seconds
Rate Limited on session_key=9488. Sleeping 6 seconds
Rate Limited on session_key=9488. Sleeping 8 seconds
Rate Limited on session_key=9488. Slee

In [191]:
placement_table.shape

(1463, 12)

##### Saving the placement_table

In [192]:
pt_path = working_directory / "data" / "clean" / "placement_table.csv"
placement_table.to_csv(pt_path)

### Merging the basic Race Dimension table via OpenF1 to each existing df

In [193]:
#merging the basic race_dim to each df
for i in range(0, len(dfs)): 
    dfs[i] = dfs[i].merge(race_dim, how="left", on=["race", "year"])


In [194]:
dfs[4].tail()

,driver,race,constructor,year,race_name,start_date,location,country_code,meeting_key,month,start_epoch
1601,BOR,3,AUD,2026,Japanese Grand Prix,2026-03-29 05:00:00+00:00,Suzuka,JPN,1281,3,1774760400
1602,LIN,3,RB,2026,Japanese Grand Prix,2026-03-29 05:00:00+00:00,Suzuka,JPN,1281,3,1774760400
1603,COL,3,ALP,2026,Japanese Grand Prix,2026-03-29 05:00:00+00:00,Suzuka,JPN,1281,3,1774760400
1604,PER,3,CAD,2026,Japanese Grand Prix,2026-03-29 05:00:00+00:00,Suzuka,JPN,1281,3,1774760400
1605,BOT,3,CAD,2026,Japanese Grand Prix,2026-03-29 05:00:00+00:00,Suzuka,JPN,1281,3,1774760400


In [195]:
for df in dfs:
    print(df.head())

  driver  race  price  year           race_name                start_date  \
0    VER     1   26.9  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   
1    HAM     1   23.7  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   
2    LEC     1   21.2  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   
3    PER     1   18.0  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   
4    RUS     1   18.6  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   

  location country_code  meeting_key  month  start_epoch  
0   Sakhir          BRN         1141      3   1678028400  
1   Sakhir          BRN         1141      3   1678028400  
2   Sakhir          BRN         1141      3   1678028400  
3   Sakhir          BRN         1141      3   1678028400  
4   Sakhir          BRN         1141      3   1678028400  
  driver  race  points  year           race_name                start_date  \
0    VER     1    35.0  2023  Bahrain Grand Prix 2023-03-05 15:00:00+00:00   
1    PER     1    28.0  202

### Dealing with Current Week

- the race_dim from the API has no data on current week, will have to rel

### Checking Nulls/NA's within the Roster
- these are racers who raaced in a given year but did not drive for the given race

In [196]:
driver_roster_long.isna().sum()

driver           0
race             0
constructor    140
year             0
dtype: int64

In [197]:
driver_roster_long = driver_roster_long.dropna(subset=["constructor"]) #this will drop records where the driver didnt actually race
driver_roster_long = driver_roster_long[driver_roster_long["constructor"] != ""]
driver_roster_long.isna().sum()
dfs[4] = driver_roster_long #saving back into the list

### Saving each file created seperately for later use

#### Team Config

In [198]:
raw_file_path = working_directory / "data" / "raw" / "rawHistPointAndPrice.xlsx"

#read in from excel
team_config = pd.read_excel(raw_file_path, sheet_name = "2026TeamConfig")
#save as its own csv
file_path_team_config = working_directory / "data" / "myTeam" / "teamConfig.csv"
team_config.to_csv(file_path_team_config)

#### Driver and Constructor Points and Prices

In [199]:
output_paths = ["driver_price", "driver_points", "constructor_price", "constructor_points", "driver_roster"]

for i in range(0, len(dfs)):
    file_path = working_directory / "data" / "clean" / f"{output_paths[i]}.csv"
    dfs[i].to_csv(file_path)

#### Verifying current week is within each df

In [200]:
for i in range(0, len(dfs)):
    print(dfs[i].tail(5))

     driver  race  price  year           race_name                start_date  \
1719    BOR     4    5.8  2026  Bahrain Grand Prix 2026-04-12 15:00:00+00:00   
1720    LIN     4    7.6  2026  Bahrain Grand Prix 2026-04-12 15:00:00+00:00   
1721    COL     4    7.6  2026  Bahrain Grand Prix 2026-04-12 15:00:00+00:00   
1722    PER     4    7.0  2026  Bahrain Grand Prix 2026-04-12 15:00:00+00:00   
1723    BOT     4    4.5  2026  Bahrain Grand Prix 2026-04-12 15:00:00+00:00   

     location country_code  meeting_key  month  start_epoch  
1719   Sakhir          BRN         1282      4   1776006000  
1720   Sakhir          BRN         1282      4   1776006000  
1721   Sakhir          BRN         1282      4   1776006000  
1722   Sakhir          BRN         1282      4   1776006000  
1723   Sakhir          BRN         1282      4   1776006000  
     driver  race  points  year           race_name                start_date  \
1671    BOR     4     NaN  2026  Bahrain Grand Prix 2026-04-12 15:

In [162]:
for df in dfs:
    print(df.shape)

(1702, 11)
(1654, 11)
(733, 11)
(757, 11)
(1466, 4)
